In [ ]:
import glob 
import cv2 
import os
import numpy as np
import models.pidnet as pidnet 
import torch
import torch.nn.functional as F 
from PIL import Image

In [ ]:
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

color_map = [(128, 64,128),
             (244, 35,232),
             ( 70, 70, 70),
             (102,102,156),
             (190,153,153),
             (153,153,153),
             (250,170, 30),
             (220,220,  0),
             (107,142, 35),
             (152,251,152),
             ( 70,130,180),
             (220, 20, 60),
             (255,  0,  0),
             (  0,  0,142),
             (  0,  0, 70),
             (  0, 60,100),
             (  0, 80,100),
             (  0,  0,230),
             (119, 11, 32)]

def input_transform(image):
    image = image.astype(np.float32)[:, :, ::-1]
    image = image / 255.0
    image -= mean
    image /= std
    return image

def load_pretrained(model, pretrained):
    pretrained_dict = torch.load(pretrained, map_location='cpu')
    if 'state_dict' in pretrained_dict:
        pretrained_dict = pretrained_dict['state_dict']
    model_dict = model.state_dict()
    
    pretrained_dict = {k[6:]: v for k, v in pretrained_dict.items() if (k[6:] in model_dict and v.shape == model_dict[k[6:]].shape)}
    msg = 'Loaded {} parameters!'.format(len(pretrained_dict))
    print('Attention!!!')
    print(msg)
    print('Over!!!')
    
    model_dict.update(pretrained_dict)
    model.load_state_dict(model_dict, strict = False)

    return model

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

folder_name = "Depth_001"
a = 'pidnet-l' 
c = True 
p = 'pretrained_models/cityscapes/PIDNet_L_Cityscapes_test.pt' 
r = f'samples/{folder_name}/' 
t = 'png' 

images_list = glob.glob(r + '*_left.' +t)
sv_path = r+'outputs/'

model = pidnet.get_pred_model(a, 19 if c else 11)
model = load_pretrained(model, p).to(device)
model.eval() 

pred_dict = {}

with torch.no_grad(): 
    for img_path in images_list:
        img_name = img_path.split("\\")[-1] 
        img = cv2.imread(os.path.join(r, img_name), cv2.IMREAD_COLOR) 
        sv_img = np.zeros_like(img).astype(np.uint8) 
        img = input_transform(img) 
        img = img.transpose((2, 0, 1)).copy() 
        img = torch.from_numpy(img).unsqueeze(0).to(device) 

        pred = model(img)
        pred = F.interpolate(pred, size=img.size()[-2:], mode='bilinear', align_corners=True)
        pred = torch.argmax(pred, dim=1).squeeze(0).cpu().numpy()
        
        for i, color in enumerate(color_map):
            for j in range(3):
                sv_img[:,:,j][pred==i] = color_map[i][j]
        sv_img = Image.fromarray(sv_img)
        
        if not os.path.exists(sv_path):
            os.mkdir(sv_path)

        pred_dict[img_name] = pred

Using device: cuda
Attention!!!
Loaded 519 parameters!
Over!!!


In [ ]:
import os
import cv2
import numpy as np

ROAD = 0
SIDEWALK = 1

def make_ground_mask(pred):
    ground_classes = [ROAD, SIDEWALK]
    mask = np.isin(pred, ground_classes).astype(np.uint8) 

    return mask

def refine_ground_mask(mask):
    kernel = np.ones((5, 5), np.uint8) 
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel) 
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel) 
    return mask

def keep_bottom_region(mask, ratio=0.7):
    h, w = mask.shape 
    top = int(h * (1 - ratio))
    new_mask = np.zeros_like(mask) 
    new_mask[top:, :] = mask[top:, :] 
    return new_mask

def keep_largest_component(mask):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    if num_labels <= 1:
        return mask

    largest_label = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    largest_mask = (labels == largest_label).astype(np.uint8)
    return largest_mask

def mask_to_image(mask):
    return (mask * 255).astype(np.uint8) 


ground_mask_dict = {}
save_dir = os.path.join(r, "ground_mask")
os.makedirs(save_dir, exist_ok=True)

for img_name, pred in pred_dict.items():
    ground_mask = make_ground_mask(pred)
    ground_mask = refine_ground_mask(ground_mask)
    ground_mask = keep_bottom_region(ground_mask, ratio=0.7)
    ground_mask = keep_largest_component(ground_mask)
    ground_mask_dict[img_name] = ground_mask

In [ ]:
import configparser

def load_camera_params(conf_path, cam_section="LEFT_CAM_FHD"):
    config = configparser.ConfigParser()
    config.read(conf_path)

    fx = float(config[cam_section]["fx"])
    baseline = float(config["STEREO"]["BaseLine"])
    return fx, baseline

def disparity16_to_float(disp16):
    return disp16.astype(np.float32) / 16.0

def disparity_to_depth(disp, fx, baseline):
    depth = np.zeros_like(disp, dtype=np.float32) 
    valid = disp > 0 
    depth[valid] = (fx * baseline) / disp[valid] 
    return depth

def depth_to_image(depth):
    valid = depth > 0
    depth_img = np.zeros_like(depth, dtype=np.uint8)

    if np.any(valid):
        d_min = depth[valid].min()
        d_max = depth[valid].max()

        if d_max > d_min:
            norm = (depth - d_min) / (d_max - d_min)
            depth_img = (norm * 255).astype(np.uint8)

    return depth_img

In [7]:
depth_dict = {}

depth_save_dir = os.path.join(r, "depth_map")
os.makedirs(depth_save_dir, exist_ok=True)

conf_path = os.path.join(r, f"{folder_name}.conf")
fx, baseline = load_camera_params(conf_path, cam_section="LEFT_CAM_FHD")

for img_name in ground_mask_dict.keys():
    disp_name = img_name.replace(".jpg", "_disp16.png").replace(".png", "_disp16.png").replace("_left", "")
    disp_path = os.path.join(r, disp_name)

    disp16 = cv2.imread(disp_path, cv2.IMREAD_UNCHANGED)
    disp = disparity16_to_float(disp16)
    depth = disparity_to_depth(disp, fx, baseline)

    depth_dict[img_name] = depth
    # TEMP : cv2.imwrite(os.path.join(depth_save_dir, img_name), depth_to_image(depth))

In [ ]:
def load_confidence_mask(conf_path):
    conf_mask = cv2.imread(conf_path, cv2.IMREAD_GRAYSCALE)
    if conf_mask is None:
        return None

    return (conf_mask > 0).astype(np.uint8)

def load_intrinsics(conf_path, cam_section="LEFT_CAM_FHD"):
    config = configparser.ConfigParser()
    config.read(conf_path)

    fx = float(config[cam_section]["fx"])
    fy = float(config[cam_section]["fy"])
    cx = float(config[cam_section]["cx"])
    cy = float(config[cam_section]["cy"])
    return fx, fy, cx, cy

def depth_to_point_cloud(depth, mask, fx, fy, cx, cy):
    h, w = depth.shape

    u, v = np.meshgrid(np.arange(w), np.arange(h)) 


    valid = (depth > 0) & (mask > 0)

    z = depth[valid]
    x = (u[valid] - cx) * z / fx 

    y = (v[valid] - cy) * z / fy 


    points = np.stack([x, y, z], axis=1).astype(np.float32)
    return points

point_cloud_dict = {} 

fx, fy, cx, cy = load_intrinsics(conf_path, cam_section="LEFT_CAM_FHD")

for img_name in depth_dict.keys():
    depth = depth_dict[img_name]
    ground_mask = ground_mask_dict[img_name]

    conf_name = img_name.replace("_left.png", "_confidence.png")
    conf_mask_path = os.path.join(r, conf_name)
    confidence_mask = load_confidence_mask(conf_mask_path)

    if confidence_mask is None:
        continue
    
    combined_mask = (ground_mask > 0) & (confidence_mask > 0)
    combined_mask = combined_mask.astype(np.uint8)

    points = depth_to_point_cloud(depth, combined_mask, fx, fy, cx, cy)
    point_cloud_dict[img_name] = points

In [ ]:
from sklearn.linear_model import LinearRegression, RANSACRegressor

def fit_plane_ransac(points): 
    X = points[:, :2]   
    y = points[:, 2]    
    ransac = RANSACRegressor( 
        estimator=LinearRegression(),
        min_samples=3, 
        residual_threshold=30.0, 
        max_trials=200, 
        random_state=42
    )

    ransac.fit(X, y) 
    p, q = ransac.estimator_.coef_ 
    r = ransac.estimator_.intercept_ 
    normal = np.array([-p, -q, 1.0], dtype=np.float32)
    normal = normal / np.linalg.norm(normal)
    inlier_mask = ransac.inlier_mask_
    inlier_points = points[inlier_mask]

    return {
        "model": ransac,
        "plane_coeff": (p, q, r),
        "normal": normal,
        "inlier_mask": inlier_mask,
        "inlier_points": inlier_points
    }

plane_dict = {}

for img_name, points in point_cloud_dict.items():
    if len(points) < 3:
        continue

    plane_result = fit_plane_ransac(points)
    plane_dict[img_name] = plane_result

In [10]:
import csv

def normal_to_slope_deg(normal):

    normal = normal.astype(np.float32)
    normal = normal / np.linalg.norm(normal)

    y_axis = np.array([0.0, 1.0, 0.0], dtype=np.float32)

    cos_theta = np.dot(normal, y_axis)
    cos_theta = np.clip(np.abs(cos_theta), 0.0, 1.0)

    slope_deg = np.degrees(np.arccos(cos_theta))

    return float(slope_deg)


def extract_prefix(img_name):

    prefix = os.path.splitext(img_name)[0]

    for suffix in [
        "_L",
        "_R",
        "_left",
        "_right",
        "_disp16",
        "_disp",
        "_confidence",
        "_confidence_save"
    ]:
        if prefix.endswith(suffix):
            prefix = prefix[:-len(suffix)]
            break

    return prefix


csv_path = os.path.join(r, "slope_result.csv")
folder_name = os.path.basename(os.path.normpath(r))

rows = []

for img_name, plane_result in plane_dict.items():

    normal = plane_result.get("normal", None)

    if normal is None:
        continue

    slope_avg = normal_to_slope_deg(normal)

    prefix = extract_prefix(img_name)

    img_path = os.path.join(r, img_name)

    rows.append([
        folder_name,
        prefix,
        img_path,
        slope_avg
    ])

with open(csv_path, "w", newline="", encoding="utf-8-sig") as f:

    writer = csv.writer(f)

    writer.writerow([
        "folder",
        "prefix",
        "path",
        "slope_avg"
    ])

    writer.writerows(rows)